In [2]:
###Locating the Planner, Router and Graph Query Files

from pathlib import Path

files = [
    Path("src/utas_research_assistant/query/planner.py"),
    Path("src/utas_research_assistant/query/reasoning_router.py"),
    Path("src/utas_research_assistant/query/models.py"),
    Path("src/utas_research_assistant/graph/queries.py"),
    Path("src/utas_research_assistant/graph/namespaces.py"),
]

for file in files:
    print(file, "->", file.exists())

src/utas_research_assistant/query/planner.py -> True
src/utas_research_assistant/query/reasoning_router.py -> True
src/utas_research_assistant/query/models.py -> True
src/utas_research_assistant/graph/queries.py -> True
src/utas_research_assistant/graph/namespaces.py -> True


In [4]:
###Running the Actual Project Planner on a Real Question
%pip install PyMuPDF -q
import sys
import json

sys.path.insert(0, "src")

from utas_research_assistant.query.planner import ReasoningPlanner

question = "Which projects are supervised by Shuxiang Xu?"

planner = ReasoningPlanner()
plan = planner.plan(question)

print("QUESTION:")
print(question)

print("\nPLANNER OUTPUT:")
print(json.dumps(plan.model_dump(), indent=2, default=str))

Note: you may need to restart the kernel to use updated packages.
QUESTION:
Which projects are supervised by Shuxiang Xu?

PLANNER OUTPUT:
{
  "method": "graph",
  "scope": "projects",
  "search_query": "Shuxiang Xu projects",
  "degree_type": null,
  "student_type": null,
  "location": null,
  "funding_status": null,
  "status": null,
  "research_category": null,
  "supervisor": "Shuxiang Xu",
  "project_id": null,
  "graph_operation": "projects_by_supervisor",
  "intent": "supervisor_project_lookup",
  "confidence": 0.9,
  "planner_method": "llm"
}


In [6]:
###Running the Actual Router on the Same Question
%pip install rank_bm25 -q


from utas_research_assistant.service import create_service

service = create_service()

router_output = service.router.route(question)

print("ROUTER OUTPUT:")
print(json.dumps(router_output, indent=2, default=str))

Note: you may need to restart the kernel to use updated packages.
ROUTER OUTPUT:
{
  "original_question": "Which projects are supervised by Shuxiang Xu?",
  "reasoning_method": "graph",
  "scope": "projects",
  "planner_method": "llm",
  "interpreted_intent": "supervisor_project_lookup",
  "extracted_constraints": {
    "supervisor": "Doctor Shuxiang Xu"
  },
  "graph_operation_used": "projects_by_supervisor",
  "candidate_count_before": 219,
  "candidate_count_after": 2,
  "ranked_retrieval_evidence": [],
  "graph_result": [
    {
      "project_id": "12259",
      "title": "Multimodal Deep Learning for Phishing Detection",
      "source_url": "https://www.utas.edu.au/research/degrees/available-projects?id=12259",
      "description": "Multimodal Deep Learning for Phishing Detection",
      "scholarship_text": "$34,315 pa",
      "closing_date": "1 October 2026",
      "degree_types": [
        "Master by Research",
        "PhD"
      ],
      "student_types": [
        "Domestic",
 

In [7]:
###Comparing the Deterministic Fallback with the LLM Planner

import json

from utas_research_assistant.query.planner import (
    ReasoningPlanner,
    fallback_reasoning_plan,
)

question = "Which projects are supervised by Shuxiang Xu?"

# Rule-based plan
fallback_plan = fallback_reasoning_plan(question)

# Normal planner: tries Qwen/Ollama and validates the result
planner = ReasoningPlanner()
final_plan = planner.plan(question)

print("QUESTION:")
print(question)

print("\n1. DETERMINISTIC FALLBACK PLAN:")
print(json.dumps(fallback_plan.model_dump(), indent=2, default=str))

print("\n2. FINAL PLANNER PLAN:")
print(json.dumps(final_plan.model_dump(), indent=2, default=str))

QUESTION:
Which projects are supervised by Shuxiang Xu?

1. DETERMINISTIC FALLBACK PLAN:
{
  "method": "graph",
  "scope": "projects",
  "search_query": "Which supervised by",
  "degree_type": null,
  "student_type": null,
  "location": null,
  "funding_status": null,
  "status": null,
  "research_category": null,
  "supervisor": "Shuxiang Xu",
  "project_id": null,
  "graph_operation": "projects_by_supervisor",
  "intent": "supervisor_project_lookup",
  "confidence": 0.45,
  "planner_method": "fallback"
}

2. FINAL PLANNER PLAN:
{
  "method": "graph",
  "scope": "projects",
  "search_query": "Shuxiang Xu projects",
  "degree_type": null,
  "student_type": null,
  "location": null,
  "funding_status": null,
  "status": null,
  "research_category": null,
  "supervisor": "Shuxiang Xu",
  "project_id": null,
  "graph_operation": "projects_by_supervisor",
  "intent": "supervisor_project_lookup",
  "confidence": 0.9,
  "planner_method": "llm"
}


In [8]:
###Inspecting the Actual Fallback Routing Rules

import inspect

print(inspect.getsource(fallback_reasoning_plan))

def fallback_reasoning_plan(question: str) -> ReasoningPlan:
    """Small rules for clear cases; ambiguous questions remain retrieval-first."""
    base = fallback_plan(question)
    text = question.casefold()
    values = base.model_dump(exclude={"scope", "search_query", "intent", "confidence", "planner_method"})
    project_id = base.project_id
    has_supervisor = bool(re.search(r"\bsupervisor\w*\b", text))
    has_project = bool(re.search(r"\bprojects?\b", text))
    if values.get("status") is None and has_project and re.search(r"\bopen\b", text):
        values["status"] = "Applications open"
    research_category = bool(re.search(r"\bresearch categor(?:y|ies)\b", text))
    profile_question = bool(re.search(r"\b(tell me about|research interests?|research background|which school|affiliated|profile|who is)\b", text))
    operation = None
    method = "retrieval"
    intent = "general_information"
    scope = base.scope

    local_context = bool(re.search(
        r"\b(?:applicant\s

In [9]:
###Tracing the Router Decision

from utas_research_assistant.service import create_service

service = create_service()

router_output = service.router.route(question)

print("Planner chose:")
print("method =", final_plan.method)
print("graph_operation =", final_plan.graph_operation)

print("\nRouter actually used:")
print("reasoning_method =", router_output.get("reasoning_method"))
print("graph_operation_used =", router_output.get("graph_operation_used"))
print("interpreted_intent =", router_output.get("interpreted_intent"))

print("\nGraph result:")
print(router_output.get("graph_result"))

Planner chose:
method = graph
graph_operation = projects_by_supervisor

Router actually used:
reasoning_method = graph
graph_operation_used = projects_by_supervisor
interpreted_intent = supervisor_project_lookup

Graph result:
[{'project_id': '12259', 'title': 'Multimodal Deep Learning for Phishing Detection', 'source_url': 'https://www.utas.edu.au/research/degrees/available-projects?id=12259', 'description': 'Multimodal Deep Learning for Phishing Detection', 'scholarship_text': '$34,315 pa', 'closing_date': '1 October 2026', 'degree_types': ['Master by Research', 'PhD'], 'student_types': ['Domestic', 'International'], 'research_categories': ['Information and Communication Technology'], 'funding_status': 'funded', 'status': 'Applications open', 'location': 'Launceston', 'primary_supervisor': 'Doctor Shuxiang Xu', 'supervisor_profile': {'canonical_name': 'Doctor Shuxiang Xu', 'title': 'Doctor', 'school': 'University of Tasmania', 'bio': 'Dr Shuxiang Xu is a Lecturer within the School of

In [10]:
###Inspecting the Router Dispatch Code

import inspect

from utas_research_assistant.query.reasoning_router import execute_graph_operation

print(inspect.getsource(execute_graph_operation))

def execute_graph_operation(graph: Graph, plan: ReasoningPlan):
    """Dispatch only named, reviewed operations; no query text comes from the LLM."""
    operation = plan.graph_operation
    if operation not in GRAPH_OPERATIONS:
        raise ValueError(f"Unsupported graph operation: {operation!r}")
    filters = _project_constraints(plan)
    if operation == "project_by_id":
        return get_project_by_id(graph, plan.project_id or "")
    if operation == "projects_by_supervisor":
        if not plan.supervisor:
            return []
        return get_projects_by_supervisor(graph, plan.supervisor)
    if operation == "multi_constraint_projects":
        return get_projects_by_constraints(graph, **filters)
    if operation == "supervisors_with_multiple_projects":
        return supervisors_with_multiple_projects(graph)
    if operation == "supervisors_with_funded_ict_international_projects":
        return supervisors_with_funded_ict_international_projects(graph)
    if operation == 

In [11]:
###Inspecting the Exact Graph Query Function

import inspect

from utas_research_assistant.graph.queries import get_projects_by_supervisor

print(inspect.getsource(get_projects_by_supervisor))

def get_projects_by_supervisor(graph: Graph, supervisor_name: str) -> list[dict]:
    matches = resolve_supervisor_names(graph, supervisor_name)
    if len(matches) > 1:
        raise SupervisorAmbiguityError(supervisor_name, matches)
    if not matches:
        return []
    return _find_projects(graph, {"supervisor": matches[0]})



In [12]:
###Inspecting the Actual Project Graph Query

from utas_research_assistant.graph.queries import _find_projects

print(inspect.getsource(_find_projects))

def _find_projects(graph: Graph, filters: dict[str, str | None] | None = None,
                   project_id: str | None = None) -> list[dict]:
    patterns = []
    bindings = {}
    if project_id is not None:
        patterns.append("FILTER(STR(?projectId) = ?wantedProjectId)")
        bindings["wantedProjectId"] = Literal(str(project_id))
    for index, (field, requested) in enumerate((filters or {}).items()):
        if requested is None:
            continue
        if field == "project_id":
            patterns.append(f"FILTER(STR(?projectId) = ?wanted{index})")
            bindings[f"wanted{index}"] = Literal(str(requested))
            continue
        if field not in RELATIONS:
            raise ValueError(f"Unknown graph filter: {field}")
        predicate, _ = RELATIONS[field]
        variable = f"entity{index}"
        label_variable = f"label{index}"
        patterns.append(f"?project utas:{predicate} ?{variable} . ?{variable} rdfs:label ?{label_variable} .")
        patte

In [13]:
###Inspecting How Projects Are Actually Queried

import inspect
from utas_research_assistant.graph.queries import _find_projects

print(inspect.getsource(_find_projects))

def _find_projects(graph: Graph, filters: dict[str, str | None] | None = None,
                   project_id: str | None = None) -> list[dict]:
    patterns = []
    bindings = {}
    if project_id is not None:
        patterns.append("FILTER(STR(?projectId) = ?wantedProjectId)")
        bindings["wantedProjectId"] = Literal(str(project_id))
    for index, (field, requested) in enumerate((filters or {}).items()):
        if requested is None:
            continue
        if field == "project_id":
            patterns.append(f"FILTER(STR(?projectId) = ?wanted{index})")
            bindings[f"wanted{index}"] = Literal(str(requested))
            continue
        if field not in RELATIONS:
            raise ValueError(f"Unknown graph filter: {field}")
        predicate, _ = RELATIONS[field]
        variable = f"entity{index}"
        label_variable = f"label{index}"
        patterns.append(f"?project utas:{predicate} ?{variable} . ?{variable} rdfs:label ?{label_variable} .")
        patte

In [14]:
###Tracing SPARQL Queries Executed by the Graph

graph = service.router.graph

original_query = graph.query

def traced_query(query_object, *args, **kwargs):
    print("SPARQL BEING EXECUTED:\n")
    print(str(query_object))
    print("\n" + "=" * 60 + "\n")
    return original_query(query_object, *args, **kwargs)

graph.query = traced_query

In [15]:
from utas_research_assistant.graph.queries import get_projects_by_supervisor

result = get_projects_by_supervisor(graph, "Shuxiang Xu")

print("RESULT:")
print(result)

SPARQL BEING EXECUTED:


PREFIX utas: <https://example.org/utas-research/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?project ?projectId ?title ?sourceUrl ?description ?scholarshipText ?closingDate
WHERE {
  ?project rdf:type utas:ResearchProject ; utas:projectId ?projectId ; utas:title ?title .
  OPTIONAL { ?project utas:sourceUrl ?sourceUrl }
  OPTIONAL { ?project utas:description ?description }
  OPTIONAL { ?project utas:scholarshipText ?scholarshipText }
  OPTIONAL { ?project utas:closingDate ?closingDate }
  ?project utas:supervisedBy ?entity0 . ?entity0 rdfs:label ?label0 .
  FILTER(LCASE(STR(?label0)) = ?wanted0)
}
ORDER BY ?projectId



RESULT:
[{'project_id': '12259', 'title': 'Multimodal Deep Learning for Phishing Detection', 'source_url': 'https://www.utas.edu.au/research/degrees/available-projects?id=12259', 'description': 'Multimodal Deep Learning for Phishing Detection', 'scholarship_tex

In [16]:
###Displaying the Exact Generated SPARQL for the Supervisor Query

from utas_research_assistant.graph.queries import (
    BASE,
    BASE_SELECT,
    RELATIONS,
    canonical,
)
from rdflib import Literal

filters = {"supervisor": "Shuxiang Xu"}

patterns = []
bindings = {}

for index, (field, requested) in enumerate(filters.items()):
    predicate, _ = RELATIONS[field]

    variable = f"entity{index}"
    label_variable = f"label{index}"

    patterns.append(
        f"?project utas:{predicate} ?{variable} . "
        f"?{variable} rdfs:label ?{label_variable} ."
    )

    patterns.append(
        f"FILTER(LCASE(STR(?{label_variable})) = ?wanted{index})"
    )

    bindings[f"wanted{index}"] = Literal(
        canonical(field, requested)
    )

query = BASE + BASE_SELECT % "\n  ".join(patterns)

print("GENERATED SPARQL:\n")
print(query)

print("\nBINDINGS:")
print(bindings)

GENERATED SPARQL:


PREFIX utas: <https://example.org/utas-research/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?project ?projectId ?title ?sourceUrl ?description ?scholarshipText ?closingDate
WHERE {
  ?project rdf:type utas:ResearchProject ; utas:projectId ?projectId ; utas:title ?title .
  OPTIONAL { ?project utas:sourceUrl ?sourceUrl }
  OPTIONAL { ?project utas:description ?description }
  OPTIONAL { ?project utas:scholarshipText ?scholarshipText }
  OPTIONAL { ?project utas:closingDate ?closingDate }
  ?project utas:supervisedBy ?entity0 . ?entity0 rdfs:label ?label0 .
  FILTER(LCASE(STR(?label0)) = ?wanted0)
}
ORDER BY ?projectId


BINDINGS:
{'wanted0': rdflib.term.Literal('shuxiang xu')}


In [18]:
###Finding Where the Main Project Loads the Graph

from pathlib import Path

for path in Path("../src").rglob("*.py"):
    text = path.read_text(encoding="utf-8")

    if "utas_research_graph.ttl" in text or ".parse(" in text and "turtle" in text:
        print("\nFILE:", path)
        print("-" * 70)

        for line_number, line in enumerate(text.splitlines(), 1):
            if (
                "utas_research_graph.ttl" in line
                or "Graph()" in line
                or ".parse(" in line
                or "turtle" in line
            ):
                print(line_number, ":", line)

In [19]:
###Checking Access to the Main Project Source Folder

from pathlib import Path

print("Current folder:", Path.cwd())
print("../src exists:", Path("../src").exists())

print("\nFiles inside main src:")
for path in Path("../src").rglob("*.py"):
    print(path)

Current folder: /Users/hosna/Documents/UTAS_research_guide_Agent/utas-research-degree-assistant
../src exists: False

Files inside main src:
